In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

##Load silver holidays

In [0]:
df_silver = spark.table(SILVER_HOLIDAYS)
print(f"Silver rows: {df_silver.count()}")
df_silver.show(5, truncate=40)

##Build dim_holiday

In [0]:
df_dim_holiday = (df_silver
    .withColumn("_gold_ingested_at", 
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "holiday_name",
        "holiday_type",
        "holiday_description",
        "holiday_date",
        "year",
        "country",
        "_gold_ingested_at"
    )
)

print(f"dim_holiday rows: {df_dim_holiday.count()}")

##Write to Gold

In [0]:
(df_dim_holiday
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_HOLIDAYS)
)

print(f"✅ {df_dim_holiday.count()} rows written to {GOLD_HOLIDAYS}")

##Validate

In [0]:
df_check = spark.table(GOLD_HOLIDAYS)

print(f"Total rows          : {df_check.count()}")
print(f"Years covered       : {[r[0] for r in df_check.select('year').distinct().orderBy('year').collect()]}")

df_check.select(
    "holiday_name", "holiday_date",
    "holiday_type", "country"
).show(10, truncate=40)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_holidays;